# ASAP8 VIP somatic electrophysiology characterization — multi-mouse, 50 µm depth bins

This notebook is intentionally **separate from Detection-of-Change response analysis**. It asks what electrophysiological phenotypes are present in ASAP8+ VIP somata and how those phenotypes vary with cortical depth, session, and mouse.

### Design principles
- Every ROI inherits the cortical depth of the DMD on which it was recorded (`dmd1_depth` / `dmd2_depth`).
- Cells are grouped into **50 µm half-open bins**: 0–50, 50–100, 100–150 µm, etc.
- Plot color is deliberately binary and independent of DMD: **<100 µm = `#eaa186`**, **≥100 µm = `#4379bc`**.
- Cell-level points are descriptive. Because cells are nested in sessions and mice, session-level and mouse-level summaries are shown explicitly.
- **Synchrony is only computed between simultaneously recorded neurons from the same session.** Pairwise values are then collapsed to one median per session × depth-bin pair before comparing sessions/mice, avoiding pseudo-replication from the combinatorial number of cell pairs.
- If `roi_identity_registration.csv` exists for a mouse, its `global_cell_id` is attached to the ROI table and used for optional longitudinal same-cell views. The ephys analysis does not require registration.

### ROI QC is optional
The manual registrar keeps every anatomical ROI identity and records QC status separately.

Use:

```python
APPLY_ROI_QC_FILTER = False
```

to analyze **all ROIs regardless of automated voltage QC**. This is the default.

Set it to `True` to require both the processed trace-H5 `valid_rois_mask` and the registrar's `valid_roi` flag to pass.

`excluded=True` is a separate manual annotation and is **always respected**, independent of this QC toggle.

QC fields remain in `roi_ephys_qc_manifest.csv` in either mode so inclusion can always be audited.


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from pathlib import Path
from itertools import combinations
import json
import re
import warnings

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import display, HTML

from vip_slap2_analysis.io.session_registry import VIPSessionRegistry

display(HTML("<style>.container { width:100% !important; }</style>"))

SUPERFICIAL_COLOR = "#eaa186"
DEEP_COLOR = "#4379bc"
GRAY = "#777777"
LIGHT_GRAY = "#d9d9d9"
CHARCOAL = "#2d2926"

plt.rcParams.update({
    "figure.dpi": 115,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "font.size": 10.5,
    "axes.labelsize": 11,
    "axes.titlesize": 13,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


## 1. Cohort and analysis parameters

By default, every eligible passive DoC session is included. Set `TARGET_SESSION_LABELS` to values such as `['A0', 'A1', 'B2']` if you want a targeted comparison.


In [ ]:
BASE_PATH = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics")
TARGET_MICE = [852835,863774]                 # None = all mice
TARGET_SESSION_LABELS = None       # e.g. ["A0", "A1", "B2"]; None = all
SELECTED_SESSION_IDS = None

PARADIGMS = ["change_detection_passive"]
EXCLUDE_SESSION_TYPES = ["expression_check", "volume_imaging"]
TRACE_VARIANT = "dff_robust_f0_trial"
REGISTRATION_FILENAME = "roi_identity_registration.csv"

# Automated voltage-ROI QC is advisory by default. Set True to exclude QC-failing ROIs.
APPLY_ROI_QC_FILTER = False

DEPTH_BIN_UM = 100
DEPTH_COLOR_THRESHOLD_UM = 100

DETECTION = {
    "candidate_height_sd": 1.5,
    "prominence_dff": 0.10,
    "prominence_window_ms": 50.0,
    "refractory_ms": 0.8,
    "min_width_ms": 1.0,
    "group_link_ms": 20.0,
    "trial_edge_ms": 2.0,
}
WAVEFORM_PRE_MS = 5.0
WAVEFORM_POST_MS = 60.0
ISOLATION_MS = 60.0
PLATEAU_WINDOW_MS = (8.0, 50.0)

STTC_DT_MS = 5.0
BURST_STTC_DT_MS = 20.0
COUNT_CORR_BINS_MS = (20.0, 50.0)

SAVE_FIGURES = True
SAVE_TABLES = True
FAIL_FAST = False

OUTPUT_ROOT = BASE_PATH / "ASAP8" / "analysis" / "asap8_somatic_ephys_50um"
FIG_DIR = OUTPUT_ROOT / "figures"
TABLE_DIR = OUTPUT_ROOT / "tables"
for p in (OUTPUT_ROOT, FIG_DIR, TABLE_DIR):
    p.mkdir(parents=True, exist_ok=True)


def save_panel(fig, name):
    if SAVE_FIGURES:
        for ext in ("png", "pdf", "svg"):
            fig.savefig(FIG_DIR / f"{name}.{ext}", facecolor="white")


def depth_color(depth_um):
    return SUPERFICIAL_COLOR if float(depth_um) < DEPTH_COLOR_THRESHOLD_UM else DEEP_COLOR


def depth_bin_values(depth_um):
    if not np.isfinite(depth_um):
        return np.nan, "unknown"
    start = int(np.floor(float(depth_um) / DEPTH_BIN_UM) * DEPTH_BIN_UM)
    return start, f"{start}–{start + DEPTH_BIN_UM}"


## 2. Resolve sessions and optional manual identities


In [ ]:
def asset_value(asset, name, default=None):
    value = getattr(asset, name, default)
    return value() if callable(value) else value


def make_session_label(row):
    image_set = row.get("image_set", np.nan)
    day = row.get("image_set_day_index", np.nan)
    if pd.notna(image_set) and pd.notna(day):
        return f"{image_set}{int(day)}"
    return str(row.get("session_type", row["session_id"]))


def row_depth(row, dmd):
    for key in (f"dmd{dmd}_depth", f"dmd{dmd}_depth_um", f"DMD{dmd}_depth", f"DMD{dmd}_depth_um"):
        if key in row and pd.notna(row[key]):
            return float(row[key])
    metadata = row.get("metadata", {}) if isinstance(row.get("metadata", {}), dict) else {}
    for key in (f"dmd{dmd}_depth", f"dmd{dmd}_depth_um"):
        if key in metadata and pd.notna(metadata[key]):
            return float(metadata[key])
    raise KeyError(f"No cortical depth metadata found for DMD{dmd}")


def _boolish(value, default=False):
    if pd.isna(value):
        return bool(default)
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    return str(value).strip().lower() in {"true", "1", "yes", "y"}


def load_manual_identity_table(asset):
    path = Path(asset.session_dir).parent / REGISTRATION_FILENAME
    if not path.exists():
        # No manual table: identity is session-specific; H5 QC is still retained as metadata.
        return pd.DataFrame(columns=["session_id", "dmd", "roi", "global_cell_id", "valid_roi", "excluded"])
    table = pd.read_csv(path, dtype={"session_id": str, "global_cell_id": str})
    if "excluded" not in table:
        table["excluded"] = False
    if "valid_roi" not in table:
        warnings.warn(f"{path} has no valid_roi column; treating registration validity as True.")
        table["valid_roi"] = True
    table["excluded"] = table["excluded"].map(lambda x: _boolish(x, default=False)).astype(bool)
    table["valid_roi"] = table["valid_roi"].map(lambda x: _boolish(x, default=True)).astype(bool)
    table["global_cell_id"] = table.get("global_cell_id", "").fillna("").replace("nan", "")
    return table


registry = VIPSessionRegistry.from_basepath(BASE_PATH)
kwargs = dict(paradigms=PARADIGMS, exclude_session_types=EXCLUDE_SESSION_TYPES)
if TARGET_MICE is not None:
    kwargs["subject_ids"] = list(TARGET_MICE)

session_df = registry.sessions(**kwargs).copy()
if SELECTED_SESSION_IDS is not None:
    session_df = session_df[session_df["session_id"].astype(str).isin(SELECTED_SESSION_IDS)]

session_df["session_datetime"] = pd.to_datetime(
    session_df["session_id"].astype(str).str.extract(r"(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2})")[0],
    format="%Y-%m-%d_%H-%M-%S", errors="coerce",
)
session_df = session_df.sort_values(["subject_id", "session_datetime"]).reset_index(drop=True)
session_df["session_order"] = session_df.groupby("subject_id").cumcount()
session_df["session_label"] = session_df.apply(make_session_label, axis=1)
if TARGET_SESSION_LABELS is not None:
    session_df = session_df[session_df["session_label"].astype(str).isin([str(x) for x in TARGET_SESSION_LABELS])]

session_items = []
for _, row in session_df.iterrows():
    asset = registry.resolve_assets(row)
    trace_h5 = Path(asset.derived_dir) / "voltage" / f"voltage_session_traces_{TRACE_VARIANT}.h5"
    if not trace_h5.exists():
        warnings.warn(f"{asset.session_id}: missing {trace_h5.name}; skipped")
        continue
    session_items.append({"asset": asset, "row": row.to_dict(), "trace_h5": trace_h5})

if not session_items:
    raise RuntimeError("No complete sessions found.")

display(session_df[[c for c in ["subject_id", "session_id", "session_label", "session_type"] if c in session_df]])
print(f"Resolved {len(session_items)} sessions across {len(set(str(x['asset'].subject_id) for x in session_items))} mice")


## 3. Spike/event detection and waveform metrics

The detector matches the candidate-peak scheme used in the DoC spike notebook so event counts are directly comparable. Waveform features are restricted to isolated spikes; compound/burst structure is summarized separately.


In [ ]:
def contiguous_runs(labels):
    labels = np.asarray(labels)
    changes = np.flatnonzero(np.diff(labels) != 0) + 1
    starts = np.r_[0, changes]
    stops = np.r_[changes, len(labels)]
    return [(a, b, labels[a]) for a, b in zip(starts, stops) if labels[a] > 0]


def detect_spikes(y, fs, sample_epoch=None, trial_lengths=None):
    y = np.asarray(y, dtype=float)
    finite = np.isfinite(y)
    if finite.sum() < 3:
        return np.array([], int), np.array([], object), np.array([], int)
    if not finite.all():
        y = np.interp(np.arange(len(y)), np.flatnonzero(finite), y[finite])
    segments = [(0, len(y), 1)] if sample_epoch is None else contiguous_runs(sample_epoch)
    peaks_all = []
    prominence_wlen = max(3, int(round(DETECTION["prominence_window_ms"] / 1000 * fs)))
    prominence_wlen += 1 - prominence_wlen % 2
    distance = max(1, int(round(DETECTION["refractory_ms"] / 1000 * fs)))
    min_width = max(1.0, DETECTION["min_width_ms"] / 1000 * fs)

    for start, stop, _ in segments:
        segment = y[start:stop]
        center = float(np.mean(segment))
        scale = float(np.std(segment, ddof=1))
        peaks, _ = signal.find_peaks(
            segment,
            height=center + DETECTION["candidate_height_sd"] * scale,
            prominence=DETECTION["prominence_dff"],
            distance=distance,
            width=min_width,
            wlen=prominence_wlen,
        )
        peaks_all.extend((peaks + start).tolist())

    peaks = np.asarray(sorted(peaks_all), dtype=np.int64)
    if trial_lengths is not None and DETECTION["trial_edge_ms"] > 0 and len(peaks):
        edges = np.cumsum(np.asarray(trial_lengths, dtype=int))[:-1]
        edge = int(round(DETECTION["trial_edge_ms"] / 1000 * fs))
        keep = np.ones(len(peaks), dtype=bool)
        for boundary in edges:
            keep &= np.abs(peaks - boundary) > edge
        peaks = peaks[keep]

    classes = np.full(len(peaks), "singleton", dtype=object)
    event_id = np.arange(len(peaks), dtype=int)
    if len(peaks):
        max_gap = DETECTION["group_link_ms"] / 1000 * fs
        splits = np.r_[0, np.flatnonzero(np.diff(peaks) > max_gap) + 1, len(peaks)]
        for eid, (a, b) in enumerate(zip(splits[:-1], splits[1:])):
            event_id[a:b] = eid
            classes[a:b] = "singleton" if b-a == 1 else ("doublet" if b-a == 2 else "burst")
    return peaks, classes, event_id


def crossing_time(time_ms, waveform, peak_idx, fraction, side):
    target = fraction * waveform[peak_idx]
    if side == "left":
        segment = waveform[:peak_idx+1]
        hits = np.flatnonzero(segment <= target)
        if not len(hits):
            return np.nan
        i = hits[-1]
        if i >= peak_idx:
            return time_ms[i]
        j = i + 1
    else:
        segment = waveform[peak_idx:]
        hits = np.flatnonzero(segment <= target)
        hits = hits[hits > 0]
        if not len(hits):
            return np.nan
        j = peak_idx + hits[0]
        i = j - 1
    y0, y1 = waveform[i], waveform[j]
    if y1 == y0:
        return float(time_ms[i])
    alpha = (target - y0) / (y1 - y0)
    return float(time_ms[i] + alpha * (time_ms[j] - time_ms[i]))


def waveform_summary(y, peaks, fs):
    pre = int(round(WAVEFORM_PRE_MS / 1000 * fs))
    post = int(round(WAVEFORM_POST_MS / 1000 * fs))
    iso = int(round(ISOLATION_MS / 1000 * fs))
    time_ms = np.arange(-pre, post + 1) / fs * 1000
    isolated = []
    plateau_indices = []

    for k, peak in enumerate(peaks):
        prev_gap = peak - peaks[k-1] if k > 0 else np.inf
        next_gap = peaks[k+1] - peak if k+1 < len(peaks) else np.inf
        if peak-pre < 0 or peak+post >= len(y):
            continue
        w = np.asarray(y[peak-pre:peak+post+1], float)
        baseline = np.nanmedian(w[:max(1, pre-int(round(1e-3*fs)))])
        w = w - baseline
        amp = w[pre]
        p0 = pre + int(round(PLATEAU_WINDOW_MS[0]/1000*fs))
        p1 = pre + int(round(PLATEAU_WINDOW_MS[1]/1000*fs))
        if amp > 0 and p1 <= len(w):
            plateau_indices.append(float(np.nanmean(w[p0:p1]) / amp))
        if prev_gap > iso and next_gap > iso and amp > 0:
            isolated.append(w)

    if not isolated:
        return {
            "n_isolated_waveforms": 0,
            "median_spike_amplitude_dff": np.nan,
            "median_width50_ms": np.nan,
            "median_rise10_90_ms": np.nan,
            "median_decay50_ms": np.nan,
            "median_plateau_index": np.nanmedian(plateau_indices) if plateau_indices else np.nan,
        }

    waves = np.asarray(isolated)
    med = np.nanmedian(waves, axis=0)
    peak_idx = pre
    amp = med[peak_idx]
    t10 = crossing_time(time_ms, med, peak_idx, 0.10, "left")
    t90 = crossing_time(time_ms, med, peak_idx, 0.90, "left")
    l50 = crossing_time(time_ms, med, peak_idx, 0.50, "left")
    r50 = crossing_time(time_ms, med, peak_idx, 0.50, "right")
    return {
        "n_isolated_waveforms": int(len(waves)),
        "median_spike_amplitude_dff": float(amp),
        "median_width50_ms": float(r50-l50) if np.isfinite(r50) and np.isfinite(l50) else np.nan,
        "median_rise10_90_ms": float(t90-t10) if np.isfinite(t90) and np.isfinite(t10) else np.nan,
        "median_decay50_ms": float(r50) if np.isfinite(r50) else np.nan,
        "median_plateau_index": np.nanmedian(plateau_indices) if plateau_indices else np.nan,
    }


## 4. Process every ROI and retain per-session spike trains


In [ ]:
roi_rows = []
qc_rows = []
spike_store = {}
failure_rows = []

for item in session_items:
    asset, row, trace_h5 = item["asset"], item["row"], item["trace_h5"]
    subject_id = str(asset.subject_id)
    session_id = str(asset.session_id)
    manual = load_manual_identity_table(asset)
    manual_lookup = (
        manual.drop_duplicates(["session_id", "dmd", "roi"], keep="last")
        .set_index(["session_id", "dmd", "roi"])
        if len(manual) else None
    )
    print("Processing", subject_id, row["session_label"], session_id)

    try:
        with h5py.File(trace_h5, "r") as h5:
            for dmd_key in sorted(k for k in h5 if k.startswith("DMD")):
                dmd = int(dmd_key.replace("DMD", ""))
                depth_um = row_depth(row, dmd)
                depth_start, depth_bin = depth_bin_values(depth_um)
                group = h5[dmd_key]
                timebase = np.asarray(group["timebase_sec"][:], float)
                fs = 1.0 / np.nanmedian(np.diff(timebase[:min(len(timebase), 100000)]))
                n_rois = int(group["dff"].shape[0])
                h5_valid = (
                    np.asarray(group["valid_rois_mask"][:], bool)
                    if "valid_rois_mask" in group else np.ones(n_rois, bool)
                )
                if len(h5_valid) != n_rois:
                    warnings.warn(
                        f"{session_id} {dmd_key}: valid_rois_mask has {len(h5_valid)} rows "
                        f"but dff has {n_rois}; treating all H5 ROI rows as valid."
                    )
                    h5_valid = np.ones(n_rois, bool)

                sample_epoch = group["sample_epoch"][:] if "sample_epoch" in group else None
                trial_lengths = group["trial_lengths_samples"][:] if "trial_lengths_samples" in group else None
                duration_s = len(timebase) / fs

                for roi in range(n_rois):
                    global_cell_id = ""
                    manually_registered = False
                    manual_valid = True
                    manually_excluded = False

                    if manual_lookup is not None and (session_id, int(dmd), int(roi)) in manual_lookup.index:
                        ann = manual_lookup.loc[(session_id, int(dmd), int(roi))]
                        global_cell_id = str(ann.get("global_cell_id", "") or "")
                        if global_cell_id == "nan":
                            global_cell_id = ""
                        manually_registered = bool(global_cell_id)
                        manual_valid = _boolish(ann.get("valid_roi", True), default=True)
                        manually_excluded = _boolish(ann.get("excluded", False), default=False)

                    trace_valid = bool(h5_valid[roi])
                    qc_pass = bool(trace_valid and manual_valid)
                    analysis_valid = bool((not manually_excluded) and ((not APPLY_ROI_QC_FILTER) or qc_pass))

                    qc_rows.append({
                        "subject_id": subject_id,
                        "session_id": session_id,
                        "session_label": str(row["session_label"]),
                        "session_order": int(row["session_order"]),
                        "dmd": int(dmd),
                        "roi": int(roi),
                        "depth_um": float(depth_um),
                        "depth_bin_start_um": depth_start,
                        "depth_bin": depth_bin,
                        "global_cell_id": global_cell_id,
                        "manually_registered": manually_registered,
                        "trace_h5_valid_roi": trace_valid,
                        "registration_valid_roi": bool(manual_valid),
                        "manually_excluded": bool(manually_excluded),
                        "qc_pass": qc_pass,
                        "analysis_included": analysis_valid,
                        "exclusion_reason": (
                            "manual_excluded" if manually_excluded else
                            "trace_h5_qc_fail" if APPLY_ROI_QC_FILTER and not trace_valid else
                            "registration_csv_qc_fail" if APPLY_ROI_QC_FILTER and not manual_valid else
                            ""
                        ),
                    })

                    # All observations remain in the QC manifest. Only observations excluded by the
                    # current analysis rule are skipped here.
                    if not analysis_valid:
                        continue

                    y = np.asarray(group["dff"][int(roi), :], float)
                    peaks, classes, event_ids = detect_spikes(
                        y, fs, sample_epoch=sample_epoch, trial_lengths=trial_lengths
                    )
                    spike_times = timebase[peaks]
                    key = (subject_id, session_id, int(dmd), int(roi))
                    event_counts = pd.Series(event_ids).value_counts() if len(event_ids) else pd.Series(dtype=int)
                    burst_event_ids = set(event_counts[event_counts >= 3].index.astype(int))
                    burst_onsets = np.array([
                        spike_times[np.flatnonzero(event_ids == eid)[0]]
                        for eid in sorted(burst_event_ids)
                    ], float) if burst_event_ids else np.array([], float)

                    spike_store[key] = {
                        "spike_times": np.asarray(spike_times, float),
                        "burst_onsets": burst_onsets,
                        "t_start": float(timebase[0]),
                        "t_stop": float(timebase[-1]),
                        "depth_um": depth_um,
                        "depth_bin": depth_bin,
                        "depth_bin_start_um": depth_start,
                        "dmd": int(dmd),
                    }

                    wf = waveform_summary(y, peaks, fs)
                    dy = np.diff(y[np.isfinite(y)])
                    noise = (
                        1.4826 * np.nanmedian(np.abs(dy - np.nanmedian(dy))) / np.sqrt(2)
                        if len(dy) else np.nan
                    )
                    n_events = int(len(np.unique(event_ids))) if len(event_ids) else 0
                    n_burst_spikes = int(np.sum(classes == "burst"))
                    n_compound_events = int(np.sum(event_counts >= 2)) if len(event_counts) else 0

                    roi_rows.append({
                        "subject_id": subject_id,
                        "session_id": session_id,
                        "session_label": str(row["session_label"]),
                        "session_order": int(row["session_order"]),
                        "session_type": str(row.get("session_type", "")),
                        "dmd": int(dmd),
                        "roi": int(roi),
                        "depth_um": float(depth_um),
                        "depth_bin_start_um": depth_start,
                        "depth_bin": depth_bin,
                        "depth_color_class": "<100 µm" if depth_um < 100 else "≥100 µm",
                        "global_cell_id": global_cell_id,
                        "manually_registered": manually_registered,
                        "valid_roi": qc_pass,
                        "trace_h5_valid_roi": trace_valid,
                        "registration_valid_roi": bool(manual_valid),
                        "excluded": bool(manually_excluded),
                        "analysis_included": analysis_valid,
                        "recording_duration_s": duration_s,
                        "n_spikes": int(len(peaks)),
                        "spike_rate_hz": len(peaks) / duration_s,
                        "event_rate_hz": n_events / duration_s,
                        "compound_event_fraction": n_compound_events / n_events if n_events else np.nan,
                        "burst_spike_fraction": n_burst_spikes / len(peaks) if len(peaks) else np.nan,
                        "burst_event_rate_hz": len(burst_event_ids) / duration_s,
                        "noise_dff": noise,
                        "spike_snr": (
                            wf["median_spike_amplitude_dff"] / noise
                            if np.isfinite(noise) and noise > 0 else np.nan
                        ),
                        **wf,
                    })

    except Exception as exc:
        failure_rows.append({"subject_id": subject_id, "session_id": session_id, "error": repr(exc)})
        print("  FAILED:", repr(exc))
        if FAIL_FAST:
            raise

roi_metrics = pd.DataFrame(roi_rows)
roi_qc_manifest = pd.DataFrame(qc_rows)
failures_df = pd.DataFrame(failure_rows)

print("Automated ROI QC filtering is", "ON" if APPLY_ROI_QC_FILTER else "OFF")

if roi_metrics.empty:
    raise RuntimeError("No ROI metrics were produced after the current inclusion rules.")

# Every downstream analysis object is built from observations admitted by the current inclusion rule.
analysis_roi_metrics = roi_metrics.copy()
depth_order = (
    analysis_roi_metrics[["depth_bin", "depth_bin_start_um"]]
    .drop_duplicates().sort_values("depth_bin_start_um")["depth_bin"].tolist()
)
analysis_roi_metrics["depth_bin"] = pd.Categorical(
    analysis_roi_metrics["depth_bin"], categories=depth_order, ordered=True
)

if SAVE_TABLES:
    roi_metrics.to_csv(TABLE_DIR / "roi_ephys_metrics.csv", index=False)
    roi_qc_manifest.to_csv(TABLE_DIR / "roi_ephys_qc_manifest.csv", index=False)
    failures_df.to_csv(TABLE_DIR / "session_failures.csv", index=False)

display(analysis_roi_metrics.head())
if len(roi_qc_manifest):
    qc_summary = (
        roi_qc_manifest.groupby(["analysis_included", "exclusion_reason"], dropna=False)
        .size().rename("n_roi_observations").reset_index()
    )
    display(qc_summary)
    print(
        f"Analysis inclusion: {int(roi_qc_manifest['analysis_included'].sum())} / {len(roi_qc_manifest)} "
        f"ROI observations enter physiology (APPLY_ROI_QC_FILTER={APPLY_ROI_QC_FILTER})."
    )
print(
    f"{len(analysis_roi_metrics)} included ROI observations · "
    f"{analysis_roi_metrics['session_id'].nunique()} sessions · "
    f"{analysis_roi_metrics['subject_id'].nunique()} mice"
)


## 5. Sampling QC — valid ROI observations, sessions, and mice per 50 µm bin

Depth bins with many cells from one session are not equivalent to bins replicated across animals. This table makes that distinction explicit before any metric comparison.


In [ ]:
sampling = (
    analysis_roi_metrics.groupby("depth_bin", observed=True)
    .agg(
        n_cells=("roi", "size"),
        n_sessions=("session_id", "nunique"),
        n_mice=("subject_id", "nunique"),
    )
    .reset_index()
)
display(sampling)

fig, ax = plt.subplots(figsize=(8, 4.2))
x = np.arange(len(sampling))
ax.bar(x, sampling["n_cells"], color=[depth_color(float(str(b).split("–")[0])+25) for b in sampling["depth_bin"].astype(str)])
for i, row in sampling.iterrows():
    ax.text(i, row.n_cells, f"{row.n_sessions} sess\n{row.n_mice} mice", ha="center", va="bottom", fontsize=9)
ax.set(xticks=x, xticklabels=sampling["depth_bin"].astype(str), xlabel="Depth bin (µm)", ylabel="Cells", title="Sampling across cortical depth")
fig.tight_layout(); save_panel(fig, "01_sampling_by_depth"); plt.show()


## 6. Electrophysiological phenotype across depth

Small circles are individual cells. Diamonds are **session medians** within each depth bin. The session medians are the more appropriate unit for comparing sampling blocks; the individual cells remain visible to show heterogeneity.


In [ ]:
metrics_to_plot = [
    ("spike_rate_hz", "Spike rate (Hz)"),
    ("median_spike_amplitude_dff", "Median spike amplitude (dF/F)"),
    ("median_width50_ms", "Spike FWHM (ms)"),
    ("median_rise10_90_ms", "Rise 10–90% (ms)"),
    ("compound_event_fraction", "Compound-event fraction"),
    ("median_plateau_index", "Plateau index"),
]

session_metric_summary = (
    analysis_roi_metrics.groupby(
        ["subject_id", "session_id", "session_label", "session_order", "depth_bin", "depth_bin_start_um"],
        observed=True,
    )[[m for m, _ in metrics_to_plot]]
    .median().reset_index()
)
if SAVE_TABLES:
    session_metric_summary.to_csv(TABLE_DIR / "session_depth_ephys_medians.csv", index=False)

rng = np.random.default_rng(17)
fig, axes = plt.subplots(2, 3, figsize=(14.8, 8.5))
for ax, (metric, ylabel) in zip(axes.flat, metrics_to_plot):
    for pos, bin_label in enumerate(depth_order):
        sub = analysis_roi_metrics[analysis_roi_metrics["depth_bin"].astype(str).eq(str(bin_label))]
        if len(sub):
            jitter = rng.normal(0, 0.06, len(sub))
            ax.scatter(pos+jitter, sub[metric], s=22,
                       c=[depth_color(d) for d in sub["depth_um"]], alpha=0.42, linewidth=0)
        sess = session_metric_summary[session_metric_summary["depth_bin"].astype(str).eq(str(bin_label))]
        if len(sess):
            ax.scatter(np.full(len(sess), pos), sess[metric], marker="D", s=48,
                       c=[depth_color(float(d)+DEPTH_BIN_UM/2) for d in sess["depth_bin_start_um"]],
                       edgecolor="black", linewidth=0.65, zorder=4)
    ax.set_xticks(range(len(depth_order)))
    ax.set_xticklabels(depth_order, rotation=45, ha="right")
    ax.set_xlabel("Depth bin (µm)")
    ax.set_ylabel(ylabel)
    ax.axvline(1.5 if len(depth_order) > 2 and str(depth_order[0]).startswith("0") else -10, color=LIGHT_GRAY, lw=0)
fig.suptitle("ASAP8 somatic electrophysiology across 50 µm depth bins", y=1.01)
fig.tight_layout(); save_panel(fig, "02_ephys_metrics_by_depth"); plt.show()


## 7. Within-mouse comparisons across sessions and depth

Each point below is a **session × depth-bin median**. Connecting points by mouse makes it easy to see whether depth effects repeat within an animal versus being driven by between-mouse differences.


In [ ]:
mouse_metrics = [
    ("spike_rate_hz", "Spike rate (Hz)"),
    ("median_width50_ms", "Spike FWHM (ms)"),
    ("median_plateau_index", "Plateau index"),
    ("compound_event_fraction", "Compound fraction"),
]

subjects = sorted(session_metric_summary["subject_id"].astype(str).unique())
for subject_id in subjects:
    sub = session_metric_summary[session_metric_summary["subject_id"].astype(str).eq(subject_id)].copy()
    fig, axes = plt.subplots(1, len(mouse_metrics), figsize=(15, 4.2), sharex=False)
    for ax, (metric, ylabel) in zip(axes, mouse_metrics):
        for session_id, sess in sub.groupby("session_id"):
            sess = sess.sort_values("depth_bin_start_um")
            ax.plot(sess["depth_bin_start_um"] + DEPTH_BIN_UM/2, sess[metric], "-o", color=GRAY, alpha=0.45, lw=1)
            for row in sess.itertuples():
                ax.scatter(row.depth_bin_start_um + DEPTH_BIN_UM/2, getattr(row, metric), s=48,
                           color=depth_color(row.depth_bin_start_um + DEPTH_BIN_UM/2), edgecolor="black", linewidth=0.5)
        ax.set(xlabel="Depth (µm; bin center)", ylabel=ylabel, title=ylabel)
    fig.suptitle(f"Mouse {subject_id}: session-level ephys phenotypes", y=1.02)
    fig.tight_layout(); save_panel(fig, f"03_within_mouse_{subject_id}"); plt.show()


## 8. Between-mouse summaries

Here each mouse contributes one median of its session-level medians for each depth bin. This is intentionally more conservative than pooling every ROI across mice.


In [ ]:
mouse_depth_summary = (
    session_metric_summary.groupby(["subject_id", "depth_bin", "depth_bin_start_um"], observed=True)
    [[m for m, _ in metrics_to_plot]].median().reset_index()
)
if SAVE_TABLES:
    mouse_depth_summary.to_csv(TABLE_DIR / "mouse_depth_ephys_medians.csv", index=False)

fig, axes = plt.subplots(1, len(mouse_metrics), figsize=(15, 4.2))
for ax, (metric, ylabel) in zip(axes, mouse_metrics):
    for subject_id, sub in mouse_depth_summary.groupby("subject_id"):
        sub = sub.sort_values("depth_bin_start_um")
        ax.plot(sub["depth_bin_start_um"]+DEPTH_BIN_UM/2, sub[metric], "-o", color=GRAY, alpha=0.45, lw=1)
    grand = mouse_depth_summary.groupby("depth_bin_start_um", observed=True)[metric].median().reset_index()
    ax.scatter(grand["depth_bin_start_um"]+DEPTH_BIN_UM/2, grand[metric], marker="D", s=70,
               c=[depth_color(x+DEPTH_BIN_UM/2) for x in grand["depth_bin_start_um"]], edgecolor="black", linewidth=0.7, zorder=5)
    ax.set(xlabel="Depth (µm; bin center)", ylabel=ylabel, title=ylabel)
fig.suptitle("Between-mouse depth summaries · one trajectory per mouse", y=1.02)
fig.tight_layout(); save_panel(fig, "04_between_mouse_depth_summary"); plt.show()


## 9. Session-label × depth-class comparisons

This view collapses the 50 µm bins only for visualization into the requested superficial/deep color classes. Each point remains a **session-level median**; lines join repeated sessions from the same mouse within a depth class. Use the 50 µm-bin tables above whenever the finer depth structure matters.


In [ ]:
depthclass_session_summary = (
    analysis_roi_metrics.groupby(
        ["subject_id", "session_id", "session_label", "session_order", "depth_color_class"],
        observed=True,
    )[[m for m, _ in metrics_to_plot]]
    .median().reset_index()
)
if SAVE_TABLES:
    depthclass_session_summary.to_csv(TABLE_DIR / "session_depthclass_ephys_medians.csv", index=False)

label_order = (
    depthclass_session_summary.groupby("session_label", observed=True)["session_order"]
    .median().sort_values().index.astype(str).tolist()
)
xmap = {label:i for i,label in enumerate(label_order)}
class_colors = {"<100 µm": SUPERFICIAL_COLOR, "≥100 µm": DEEP_COLOR}

fig, axes = plt.subplots(1, len(mouse_metrics), figsize=(15.2, 4.4))
for ax, (metric, ylabel) in zip(axes, mouse_metrics):
    for (subject_id, depth_class), sub in depthclass_session_summary.groupby(["subject_id", "depth_color_class"], observed=True):
        sub = sub.copy()
        sub["x"] = sub["session_label"].astype(str).map(xmap)
        sub = sub.sort_values("x")
        color = class_colors[str(depth_class)]
        ax.plot(sub["x"], sub[metric], "-o", color=color, alpha=0.42, lw=1.0, ms=4)
    # Across-mouse medians at each session label, kept separate by depth class.
    for depth_class, sub in depthclass_session_summary.groupby("depth_color_class", observed=True):
        med = sub.groupby("session_label", observed=True)[metric].median().reindex(label_order)
        xs = np.arange(len(label_order))
        ax.scatter(xs, med.values, marker="D", s=55, color=class_colors[str(depth_class)], edgecolor="black", linewidth=0.6, zorder=5)
    ax.set(xticks=np.arange(len(label_order)), xticklabels=label_order, xlabel="Session label", ylabel=ylabel)
fig.suptitle("Session-to-session ephys comparison · mouse trajectories + across-mouse medians", y=1.02)
fig.tight_layout(); save_panel(fig, "05_session_label_depthclass_ephys"); plt.show()


# Part II — within-session synchrony

Pairwise synchrony is constructed **inside each asset/session only**. No spike train is ever paired across days or mice. Raw pair values are useful for QC and matrices, but comparisons across depth/session use a single median per session × depth-bin pair.


In [ ]:
def fraction_spikes_near(a, b, dt):
    a, b = np.asarray(a, float), np.sort(np.asarray(b, float))
    if not len(a) or not len(b):
        return np.nan
    hits = 0
    for t in a:
        j = np.searchsorted(b, t)
        near = (j < len(b) and abs(b[j]-t) <= dt) or (j > 0 and abs(b[j-1]-t) <= dt)
        hits += int(near)
    return hits / len(a)


def fraction_time_tiled(times, dt, t_start, t_stop):
    times = np.sort(np.asarray(times, float))
    if not len(times) or t_stop <= t_start:
        return 0.0
    intervals = np.c_[np.maximum(times-dt, t_start), np.minimum(times+dt, t_stop)]
    intervals = intervals[intervals[:,1] > intervals[:,0]]
    if not len(intervals):
        return 0.0
    total = 0.0
    s, e = intervals[0]
    for s2, e2 in intervals[1:]:
        if s2 <= e:
            e = max(e, e2)
        else:
            total += e-s
            s, e = s2, e2
    total += e-s
    return total / (t_stop-t_start)


def sttc(a, b, dt, t_start, t_stop):
    a, b = np.asarray(a,float), np.asarray(b,float)
    if not len(a) or not len(b):
        return np.nan
    pa = fraction_spikes_near(a,b,dt); pb = fraction_spikes_near(b,a,dt)
    ta = fraction_time_tiled(a,dt,t_start,t_stop); tb = fraction_time_tiled(b,dt,t_start,t_stop)
    terms = []
    for p, t in ((pa,tb),(pb,ta)):
        den = 1-p*t
        terms.append((p-t)/den if abs(den)>1e-12 else np.nan)
    return 0.5*np.nansum(terms) if np.isfinite(terms).any() else np.nan


def binned_corr(a, b, bin_s, t_start, t_stop):
    edges = np.arange(t_start, t_stop + bin_s, bin_s)
    if len(edges) < 3:
        return np.nan
    ca = np.histogram(a, edges)[0]; cb = np.histogram(b, edges)[0]
    if np.std(ca) == 0 or np.std(cb) == 0:
        return np.nan
    return float(np.corrcoef(ca, cb)[0,1])


roi_lookup = analysis_roi_metrics.set_index(["subject_id", "session_id", "dmd", "roi"])
pair_rows = []
for subject_id, session_id in sorted({(k[0], k[1]) for k in spike_store}):
    keys = [k for k in spike_store if k[0] == subject_id and k[1] == session_id and k in roi_lookup.index]
    for ka, kb in combinations(keys, 2):
        a, b = spike_store[ka], spike_store[kb]
        t0, t1 = max(a["t_start"], b["t_start"]), min(a["t_stop"], b["t_stop"])
        if t1 <= t0:
            continue
        row_a, row_b = roi_lookup.loc[ka], roi_lookup.loc[kb]
        ba, bb = str(row_a.depth_bin), str(row_b.depth_bin)
        sa, sb = float(row_a.depth_bin_start_um), float(row_b.depth_bin_start_um)
        if (sa, ba) > (sb, bb):
            ba, bb, sa, sb = bb, ba, sb, sa
        if row_a.depth_um < 100 and row_b.depth_um < 100:
            pair_class = "superficial–superficial"
        elif row_a.depth_um >= 100 and row_b.depth_um >= 100:
            pair_class = "deep–deep"
        else:
            pair_class = "mixed-depth"
        pair_rows.append({
            "subject_id": subject_id, "session_id": session_id,
            "session_label": row_a.session_label,
            "dmd_a": ka[2], "roi_a": ka[3], "dmd_b": kb[2], "roi_b": kb[3],
            "depth_a_um": row_a.depth_um, "depth_b_um": row_b.depth_um,
            "depth_bin_pair": f"{ba} | {bb}",
            "depth_bin_pair_start_a": sa, "depth_bin_pair_start_b": sb,
            "pair_depth_class": pair_class,
            "spike_sttc_5ms": sttc(a["spike_times"], b["spike_times"], STTC_DT_MS/1000, t0, t1),
            "burst_onset_sttc_20ms": sttc(a["burst_onsets"], b["burst_onsets"], BURST_STTC_DT_MS/1000, t0, t1),
            "spike_count_corr_20ms": binned_corr(a["spike_times"], b["spike_times"], COUNT_CORR_BINS_MS[0]/1000, t0, t1),
            "spike_count_corr_250ms": binned_corr(a["spike_times"], b["spike_times"], COUNT_CORR_BINS_MS[1]/1000, t0, t1),
        })

pair_df = pd.DataFrame(pair_rows)
if len(pair_df):
    session_pair_summary = (
        pair_df.groupby(["subject_id", "session_id", "session_label", "depth_bin_pair", "depth_bin_pair_start_a", "depth_bin_pair_start_b", "pair_depth_class"], observed=True)
        [["spike_sttc_5ms", "burst_onset_sttc_20ms", "spike_count_corr_20ms", "spike_count_corr_250ms"]]
        .median().reset_index()
    )
    raw_counts = pair_df.groupby(["subject_id", "session_id", "depth_bin_pair"], observed=True).size().rename("n_raw_pairs").reset_index()
    session_pair_summary = session_pair_summary.merge(raw_counts, on=["subject_id", "session_id", "depth_bin_pair"], how="left")
else:
    session_pair_summary = pd.DataFrame()

if SAVE_TABLES:
    pair_df.to_csv(TABLE_DIR / "within_session_pairwise_synchrony.csv", index=False)
    session_pair_summary.to_csv(TABLE_DIR / "session_depthpair_synchrony_medians.csv", index=False)

display(session_pair_summary.head() if len(session_pair_summary) else session_pair_summary)


## 10. Synchrony across depths, sessions, and mice

The dots here are session medians, not individual cell pairs. This is the level at which comparisons across sessions should be read.


In [ ]:
if len(session_pair_summary):
    sync_metrics = [
        ("spike_sttc_5ms", "Spike STTC ±5 ms"),
        ("burst_onset_sttc_20ms", "Burst-onset STTC ±20 ms"),
        ("spike_count_corr_250ms", "Spike-count correlation · 250 ms"),
    ]
    class_order = ["superficial–superficial", "mixed-depth", "deep–deep"]
    class_colors = {
        "superficial–superficial": SUPERFICIAL_COLOR,
        "mixed-depth": GRAY,
        "deep–deep": DEEP_COLOR,
    }
    rng = np.random.default_rng(22)
    fig, axes = plt.subplots(1, 3, figsize=(13.8, 4.5))
    for ax, (metric, ylabel) in zip(axes, sync_metrics):
        for pos, cls in enumerate(class_order):
            sub = session_pair_summary[session_pair_summary["pair_depth_class"].eq(cls)]
            jitter = rng.normal(0, 0.05, len(sub))
            ax.scatter(pos+jitter, sub[metric], s=45, color=class_colors[cls], edgecolor="black", linewidth=0.4, alpha=0.8)
            if len(sub):
                med = np.nanmedian(sub[metric])
                ax.plot([pos-0.18,pos+0.18],[med,med], color=CHARCOAL, lw=2)
        ax.axhline(0, color=LIGHT_GRAY, lw=0.8)
        ax.set(xticks=range(3), xticklabels=["<100/<100", "mixed", "≥100/≥100"], ylabel=ylabel)
    fig.suptitle("Within-session synchrony summarized across depth classes", y=1.02)
    fig.tight_layout(); save_panel(fig, "05_synchrony_depth_classes"); plt.show()

    # Within-mouse trajectories across session labels.
    for subject_id, sub in session_pair_summary.groupby("subject_id"):
        fig, axes = plt.subplots(1, 3, figsize=(14.0, 4.2))
        labels = list(dict.fromkeys(sub.sort_values("session_id")["session_label"].astype(str)))
        xmap = {lab:i for i,lab in enumerate(labels)}
        for ax, (metric, ylabel) in zip(axes, sync_metrics):
            for cls in class_order:
                ss = sub[sub["pair_depth_class"].eq(cls)].copy()
                if not len(ss): continue
                # If a depth class has >1 bin-pair in a session, collapse once more to a session/class median.
                s2 = ss.groupby(["session_id","session_label"], observed=True)[metric].median().reset_index()
                s2["x"] = s2["session_label"].astype(str).map(xmap)
                s2 = s2.sort_values("x")
                ax.plot(s2["x"], s2[metric], "-o", color=class_colors[cls], alpha=0.8, label=cls)
            ax.axhline(0, color=LIGHT_GRAY, lw=0.8)
            ax.set(xticks=range(len(labels)), xticklabels=labels, xlabel="Session", ylabel=ylabel)
        axes[-1].legend(fontsize=8)
        fig.suptitle(f"Mouse {subject_id}: synchrony across sessions", y=1.02)
        fig.tight_layout(); save_panel(fig, f"06_synchrony_sessions_mouse_{subject_id}"); plt.show()


## 11. Optional longitudinal same-cell ephys view

This section is used only when the manual registration table supplies `global_cell_id`. It is descriptive: repeated measurements of one neuron are connected across sessions so you can distinguish longitudinal changes from cross-sectional depth differences.


In [ ]:
registered = analysis_roi_metrics[analysis_roi_metrics["manually_registered"]].copy()
tracked = registered.groupby(["subject_id", "global_cell_id"]).filter(lambda x: x["session_id"].nunique() >= 2)

if len(tracked):
    longitudinal_metrics = [
        ("spike_rate_hz", "Spike rate (Hz)"),
        ("median_width50_ms", "FWHM (ms)"),
        ("compound_event_fraction", "Compound fraction"),
        ("median_plateau_index", "Plateau index"),
    ]
    for subject_id, sub in tracked.groupby("subject_id"):
        fig, axes = plt.subplots(1, 4, figsize=(14.5, 4.0))
        labels = list(dict.fromkeys(sub.sort_values("session_order")["session_label"].astype(str)))
        xmap = {lab:i for i,lab in enumerate(labels)}
        for ax, (metric, ylabel) in zip(axes, longitudinal_metrics):
            for cell_id, cell in sub.groupby("global_cell_id"):
                cell = cell.sort_values("session_order")
                x = cell["session_label"].astype(str).map(xmap)
                ax.plot(x, cell[metric], "-o", color=depth_color(cell["depth_um"].median()), alpha=0.55, lw=1)
            ax.set(xticks=range(len(labels)), xticklabels=labels, xlabel="Session", ylabel=ylabel)
        fig.suptitle(f"Mouse {subject_id}: manually registered same-cell ephys", y=1.02)
        fig.tight_layout(); save_panel(fig, f"07_longitudinal_ephys_mouse_{subject_id}"); plt.show()
else:
    print("No manually registered cells observed in ≥2 included sessions; longitudinal panel skipped.")


## Interpretation checklist

- Treat individual ROI points as descriptive, not independent biological replicates.
- Ask whether an apparent depth trend repeats across **sessions and mice**, not merely whether pooled cells differ.
- For synchrony, use `session_depthpair_synchrony_medians.csv` for across-session comparisons. The raw pair table is QC/detail, not the inferential unit.
- A depth-bin contrast with only one contributing mouse should be reported as within-mouse evidence rather than a population-level effect.
- The manual registration table is useful for longitudinal stability but is not needed to define the cross-sectional ephys phenotype.
